# import 

In [3]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import cv2 as cv
from pathlib import Path 
import seaborn as sns
import matplotlib.pyplot as plt
from StatTools.generators.ndfnoise_generator import ndfnoise
from tqdm import tqdm
import plotly.express as px
import os
import warnings 
import gc

# make species tracks long df

In [4]:
dir_path = Path(fr'../data/knn_kalman/pyeensis')
files = list(dir_path.glob("*.csv"))
pbar = tqdm(files)
result = []
for file_path in pbar:
    df = pd.read_csv(file_path, dtype={'frame':np.uint16, 'track_id':np.uint16, 'x':np.float16, 'y':np.float16}, usecols=['frame', 'track_id', 'x', 'y'])
    df = df[(df['x']>=0)&(df['y']>=0)].reset_index(drop=True)
    df = df.round().astype({'x': np.uint16, 'y': np.uint16})
    df['file_path'] = file_path.name
    result.append(df)
df_pyeensis = pd.concat(result)
overall_pyeensis_lens = df_pyeensis.groupby(['file_path', 'track_id'])['frame'].count()

  0%|          | 0/45 [00:00<?, ?it/s]

100%|██████████| 45/45 [00:05<00:00,  7.88it/s]


In [5]:
dir_path = Path(fr'../data/knn_kalman/frufa')
files = list(dir_path.glob("*.csv"))
pbar = tqdm(files)
result = []
for file_path in pbar:
    df = pd.read_csv(file_path, dtype={'frame':np.uint16, 'track_id':np.uint16, 'x':np.float16, 'y':np.float16}, usecols=['frame', 'track_id', 'x', 'y'])
    df = df[(df['x']>=0)&(df['y']>=0)].reset_index(drop=True)
    df = df.round().astype({'x': np.uint16, 'y': np.uint16})
    df['file_path'] = file_path.name
    result.append(df)
df_frufa = pd.concat(result)
overall_frufa_lens = df_frufa.groupby(['file_path', 'track_id'])['frame'].count()

100%|██████████| 21/21 [00:03<00:00,  5.69it/s]


# make synth species tracks

## frufa

In [6]:
result = []
frame_shape = (480, 848)

pbar = tqdm(list(df_frufa.groupby('file_path')))
for file_path, file_df in pbar:
    pbar.set_description(f"Processing {file_path}")
    # get track lens
    lens = file_df.groupby('track_id')['frame'].count()
    # for each track get bound indices
    indices = lens.cumsum()
    # specify general fgn len
    pow = np.ceil(np.log2(lens.sum())).astype(int)
    # make fgn
    fgn = ndfnoise(shape = (2**pow,2), hurst=(0.94, 0.5), dtype=np.float32)
    # chop general fgn into chunks, last one is invalid tail
    chunks = np.split(fgn, indices)[:-1]
    # explode lond trackid
    idx = np.repeat(lens.index.to_list(), lens)
    #collect total df with original shell
    synth_df = file_df.sort_values(['track_id', 'frame'])
    synth_df[['x', 'y']] = np.concat(chunks, axis=0)
    synth_df[['x', 'y']] = synth_df.groupby('track_id')[['x', 'y']].cumsum().round().astype(np.int32)
    # define random start points
    start_x = np.random.randint(0, frame_shape[1], (len(lens),)).repeat(lens)
    start_y = np.random.randint(0, frame_shape[0], (len(lens),)).repeat(lens)
    start_point = np.stack([start_x, start_y], axis=1)
    synth_df[['x', 'y']] += start_point
    synth_df.to_csv(f'../data/synth_species/frufa/{file_path}')

Processing S1240004начало.csv:   0%|          | 0/21 [00:00<?, ?it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
Processing S1240005.csv:   5%|▍         | 1/21 [00:00<00:05,  3.91it/s]      /home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
Processing S1240006.csv:   5%|▍         | 1/21 [00:00<00:05,  3.91it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
Processing S1240007.csv:  14%|█▍        | 3/21 [00:00<00:04,  3.74it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_genera

## pyeensis

In [7]:
result = []
frame_shape = (720, 1280)
pbar = tqdm(list(df_pyeensis.groupby('file_path')))
for file_path, file_df in pbar:
    pbar.set_description(f"Processing {file_path}")
    # get track lens
    lens = file_df.groupby('track_id')['frame'].count()
    # for each track get bound indices
    indices = lens.cumsum()
    # specify general fgn len
    pow = np.ceil(np.log2(lens.sum())).astype(int)
    # make fgn
    fgn = ndfnoise(shape = (2**pow,2), hurst=(0.94, 0.5), dtype=np.float32)
    # chop general fgn into chunks, last one is invalid tail
    chunks = np.split(fgn, indices)[:-1]
    # explode lond trackid
    idx = np.repeat(lens.index.to_list(), lens)
    #collect total df with original shell
    synth_df = file_df.sort_values(['track_id', 'frame'])
    synth_df[['x', 'y']] = np.concat(chunks, axis=0)
    synth_df[['x', 'y']] = synth_df.groupby('track_id')[['x', 'y']].cumsum().round().astype(np.int32)
    # define random start points
    start_x = np.random.randint(0, frame_shape[1], (len(lens),)).repeat(lens)
    start_y = np.random.randint(0, frame_shape[0], (len(lens),)).repeat(lens)
    start_point = np.stack([start_x, start_y], axis=1)
    synth_df[['x', 'y']] += start_point
    synth_df.to_csv(f'../data/synth_species/pyeensis/{file_path}')


Processing S2170005.csv:   0%|          | 0/45 [00:00<?, ?it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
Processing S2170006.csv:   0%|          | 0/45 [00:00<?, ?it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
Processing S2170007.csv:   0%|          | 0/45 [00:00<?, ?it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
Processing S2170008.csv:   7%|▋         | 3/45 [00:00<00:02, 14.71it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: d